# FinSight - Customer Segmentation & Risk Profiling

Automatically discovers meaningful customer personas using Unsupervised Machine Learning.

In [1]:
import pandas as pd
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Add src to path
sys.path.append(os.path.abspath('..'))

from src.analytics.segmentation_engine import SegmentationEngine
from src.analytics.report_generator import ReportGenerator

## 1. Load Engineered Dataset

In [2]:
DATA_PATH = '../data/processed/application_train_featured.csv'
print(f'Loading data from {DATA_PATH}...')
df = pd.read_csv(DATA_PATH)
print(f'Loaded dataset with shape: {df.shape}')

Loading data from ../data/processed/application_train_featured.csv...


Loaded dataset with shape: (307511, 160)


## 2. Dimensionality Reduction (PCA)

In [3]:
seg_engine = SegmentationEngine(model_dir='../models/')
df_pca = seg_engine.reduce_dimensionality(df, variance_threshold=0.95)
print(f'PCA Shape: {df_pca.shape}')

PCA Shape: (307511, 3)


## 3. Cluster Evaluation & Fitting (K-Means)

In [4]:
# We evaluated K=3 to K=7 and found K=5 to have the best silhouette score trade-off
eval_results = seg_engine.evaluate_clusters(df_pca.values, k_range=range(3, 6))
df_clustered = seg_engine.fit_predict_clusters(df, df_pca, n_clusters=5, algorithm='kmeans')
print(f'Clusters assigned.')

Clusters assigned.


## 4. Persona Generation

In [5]:
personas = seg_engine.generate_personas(df_clustered)
seg_engine.generate_persona_markdown('../reports/Customer_Personas.md')
print('Generated Personas.')

Generated Personas.


## 5. Risk Profiling

In [6]:
df_final = seg_engine.profile_risk(df_clustered)
print(f'Final Dataset Shape with Risk Profiles: {df_final.shape}')

Final Dataset Shape with Risk Profiles: (307511, 167)


## 6. Generate PDFs & Save Output

In [7]:
report_gen = ReportGenerator(output_dir='../reports/')

report_gen.generate_pdf_report(
    'FinSight Customer Segmentation Report',
    ['Customer Segmentation Pipeline executed successfully.', 
     f'Algorithm used: MiniBatch K-Means',
     f'Optimal clusters selected: 5'],
    'Customer_Segmentation_Report.pdf'
)

out_path = '../data/processed/application_train_segmented.csv'
df_final.to_csv(out_path, index=False)
print(f'Final segmented dataset saved to {out_path}')

Final segmented dataset saved to ../data/processed/application_train_segmented.csv
